In [ ]:
# ======================================
# 🔴 CHANGE ONLY THIS SECTION
# ======================================

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

TRAIN_DIR = "/content/drive/MyDrive/Colab Notebooks/00_ANN Hyperparameter Tuning/Chapter 11 CNN Scripting/train_data"
VAL_DIR = "/content/drive/MyDrive/Colab Notebooks/00_ANN Hyperparameter Tuning/Chapter 11 CNN Scripting/val_data"

NUM_CLASSES = 5

BASE_MODEL_NAME = "vgg16"   # vgg16, resnet50, mobilenet_v2

FREEZE_LAYERS = True
FINE_TUNE = False

# ======================================
# ✅ IMPORTS
# ======================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================
# 📊 DATA PIPELINE
# ======================================

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(TRAIN_DIR, transform=transform)
val_data = datasets.ImageFolder(VAL_DIR, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=BATCH_SIZE)

# ======================================
# 🧠 LOAD PRETRAINED MODEL
# ======================================

def get_model(name):
    if name == "vgg16":
        model = models.vgg16(pretrained=True)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, NUM_CLASSES)

    elif name == "resnet50":
        model = models.resnet50(pretrained=True)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)

    elif name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=True)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

    return model

model = get_model(BASE_MODEL_NAME)
model = model.to(device)

# ======================================
# 🔒 FREEZE BASE MODEL
# ======================================

if FREEZE_LAYERS:
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze final layer
    if BASE_MODEL_NAME == "vgg16":
        for param in model.classifier[6].parameters():
            param.requires_grad = True

# ======================================
# ⚙️ LOSS & OPTIMIZER
# ======================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

# ======================================
# 🚀 TRAIN LOOP
# ======================================

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

# ======================================
# 📊 EVALUATION
# ======================================

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 157MB/s]


Epoch 1, Loss: 3.3274
Epoch 2, Loss: 3.2599
Epoch 3, Loss: 3.0710
Epoch 4, Loss: 3.1773
Epoch 5, Loss: 3.1186
Epoch 6, Loss: 3.0134
Epoch 7, Loss: 3.1132
Epoch 8, Loss: 2.8434
Epoch 9, Loss: 2.8288
Epoch 10, Loss: 2.8368
Validation Accuracy: 33.33%
